# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a ****Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.# 

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [1]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

Installing required packages...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 108.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [6]:
LLM_PROVIDER = "groq"
LLM_MODEL    = "llama-3.3-70b-versatile"

CORPUS_PATH = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

print(f"Provider : {LLM_PROVIDER}")
print(f"Model    : {LLM_MODEL}")
print(f"Corpus   : {CORPUS_PATH}")


Provider : groq
Model    : llama-3.3-70b-versatile
Corpus   : /kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus


## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [7]:
import os, json, time, csv, glob
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")


Imports loaded successfully.


## Cell 4 — API Keys + LangSmith Setup

This cell loads:
- your LLM API key
- LangSmith API key
- environment configuration

LangSmith tracing is enabled automatically for monitoring and debugging your RAG pipeline.

> Add the required API keys before running this cell.
> This section is pre-filled — no modifications needed.

In [8]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "#your project Name"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

Running on Kaggle — secrets loaded!
Environment configured successfully.


## Cell 5 — Load Documents

### Your Task

Load all policy documents from the provided corpus directory using a PDF loader.

Store the loaded documents and print the total number loaded.

In [9]:
# Discover all PDFs in the corpus directory
pdf_files = sorted(glob.glob(os.path.join(CORPUS_PATH, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files in: {CORPUS_PATH}")
for f in pdf_files:
    print(f"  {os.path.basename(f)}")

# Load each PDF individually — more robust than PyPDFDirectoryLoader
documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    documents.extend(pages)
    print(f"  Loaded {len(pages)} pages from {os.path.basename(pdf_path)}")

print("")
print(f"Total: {len(documents)} pages loaded from {len(pdf_files)} documents")


Found 11 PDF files in: /kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus
  00_Company_Profile.pdf
  01_Employee_Handbook.pdf
  02_Leave_Policy.pdf
  03_Work_From_Home_Policy.pdf
  04_Code_of_Conduct.pdf
  05_Performance_Review_Policy.pdf
  06_Compensation_and_Benefits_Policy.pdf
  07_IT_and_Data_Security_Policy.pdf
  08_Prevention_of_Sexual_Harassment_Policy.pdf
  09_Onboarding_and_Separation_Policy.pdf
  10_Travel_and_Expense_Policy.pdf
  Loaded 4 pages from 00_Company_Profile.pdf
  Loaded 4 pages from 01_Employee_Handbook.pdf
  Loaded 4 pages from 02_Leave_Policy.pdf
  Loaded 3 pages from 03_Work_From_Home_Policy.pdf
  Loaded 3 pages from 04_Code_of_Conduct.pdf
  Loaded 4 pages from 05_Performance_Review_Policy.pdf
  Loaded 4 pages from 06_Compensation_and_Benefits_Policy.pdf
  Loaded 3 pages from 07_IT_and_Data_Security_Policy.pdf
  Loaded 3 pages from 08_Prevention_of_Sexual_Harassment_Policy.pdf
  Loaded 4 pages from 09_Onboarding_and_Separation_Pol

## Cell 6 — Chunk Documents

### Your Task

Split the loaded documents into smaller chunks using `RecursiveCharacterTextSplitter`.

Store the generated chunks and print the total number created.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documents)

assert len(chunks) > 0, "No chunks created — check PDF loading in the previous cell!"

avg_size = sum(len(c.page_content) for c in chunks) // len(chunks)
print(f"Created {len(chunks)} chunks from {len(documents)} pages")
print(f"Average chunk size: {avg_size} chars")
print("Sample chunk preview:")
print(chunks[0].page_content[:300])


Created 110 chunks from 39 pages
Average chunk size: 652 chars
Sample chunk preview:
Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Company Profile
 Document Code
 ZDL-CORP-001
 Version
 V.01
 Effective Date
 01 April 2025
 Document Owner
 Corporate Communications
COMPANY OVE


## Cell 7 — Embeddings

### Your Task

Initialize an embedding model to convert document chunks into vector representations.

Use `HuggingFaceEmbeddings` for the implementation below.

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

# all-MiniLM-L6-v2: fast, zero extra API key, great for policy retrieval
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model initialized: sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized: sentence-transformers/all-MiniLM-L6-v2


## Cell 8 — Vector Store + Retriever

### Your Task

Build a FAISS vector store using the generated chunks and embeddings.

Then create a retriever for retrieving relevant document chunks during question answering.

In [12]:
from langchain_community.vectorstores import FAISS

# Build FAISS index from document chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

# MMR retriever: diverse + relevant chunks, avoids near-duplicate returns
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,           # return 5 chunks
        "fetch_k": 20,    # candidate pool before MMR re-ranking
        "lambda_mult": 0.6,  # 0=max diversity, 1=max relevance
    },
)

print(f"FAISS vector store built with {vectorstore.index.ntotal} vectors")
print("MMR retriever ready (k=5, fetch_k=20, lambda_mult=0.6)")


FAISS vector store built with 110 vectors
MMR retriever ready (k=5, fetch_k=20, lambda_mult=0.6)


## Cell 9 — LLM Initialization

The language model is initialized automatically using the configuration from Cell 2.

You may optionally adjust parameters such as:
- `temperature`
- `max_tokens`

based on your preferred response style and performance.

> This cell is pre-filled — modify only if needed.

In [13]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

LLM initialized.


## Cell 10 — Build the RAG Chain

### Your Task

Implement the RAG pipeline using LCEL and enable LangSmith tracing.

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR Help Desk assistant for Zyro Dynamics (also known as Acrux Dynamics).
Answer the employee question using ONLY the information in the HR policy context below.
Be specific: include exact numbers, dates, limits, and percentages from the context.
Do NOT add information not found in the context.
If the context does not contain enough information, say:
"I don't have enough information in the HR policy documents to answer this question."

HR Policy Context:
{context}

Employee Question: {question}

Answer:"""
)

def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        src  = doc.metadata.get("source", "").split("/")[-1]
        page = doc.metadata.get("page", "?")
        parts.append(f"[Source {i}: {src}, p.{page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

@traceable(name="zyro-rag-chain")
def rag_chain(question: str) -> str:
    docs    = retriever.invoke(question)
    context = format_docs(docs)
    prompt  = RAG_PROMPT.invoke({"context": context, "question": question})
    answer  = llm.invoke(prompt)
    return StrOutputParser().invoke(answer)

print("RAG pipeline initialized.")
print(f"  Prompt   : RAG_PROMPT (context + question)")
print(f"  Retriever: MMR FAISS (k=5)")
print(f"  LLM      : {LLM_MODEL}")


RAG pipeline initialized.
  Prompt   : RAG_PROMPT (context + question)
  Retriever: MMR FAISS (k=5)
  LLM      : llama-3.3-70b-versatile


## Cell 11 — Guardrails

### Your Task

Implement handling for out-of-scope questions before routing queries through the RAG pipeline.

In [15]:
OOS_PROMPT = ChatPromptTemplate.from_template(
    """You are a strict classifier. Decide if the question can be answered using
Zyro Dynamics (also known as Acrux Dynamics) internal HR policy documents.

HR policy documents cover ONLY these topics:
- Leave policies (earned, sick, maternity, paternity, etc.)
- Work from home / hybrid / remote arrangements
- Employee code of conduct and ethics
- Performance reviews, PIP, APR, promotions
- Compensation, salary bands, bonuses, payroll dates
- Health insurance and employee benefits
- IT and data security policies
- Prevention of sexual harassment (POSH)
- Onboarding, probation, separation, full and final settlement
- Business travel and expense reimbursement
- Company overview, grade structure, office policies

Question: {question}

Reply with exactly one word — IN_SCOPE or OUT_OF_SCOPE. No explanation."""
)

REFUSAL_MESSAGE = (
    "I'm sorry, but I can only answer HR-related questions based on "
    "Zyro Dynamics internal policy documents. Your question falls outside "
    "the scope of what I can help with here. Please reach out to the relevant "
    "department or consult an external resource."
)

_parser = StrOutputParser()

@traceable(name="zyro-ask-bot")
def ask_bot(question: str) -> dict:
    # Layer 1: LLM scope classifier
    verdict = _parser.invoke(
        llm.invoke(OOS_PROMPT.invoke({"question": question}))
    ).strip().upper()

    if "OUT_OF_SCOPE" in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": [], "in_scope": False}

    # Layer 2: FAISS relevance score floor
    docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
        question, k=5
    )

    if not docs_with_scores or docs_with_scores[0][1] < 0.25:
        return {"answer": REFUSAL_MESSAGE, "sources": [], "in_scope": False}

    docs    = [doc for doc, _ in docs_with_scores]
    context = format_docs(docs)
    prompt  = RAG_PROMPT.invoke({"context": context, "question": question})
    answer  = _parser.invoke(llm.invoke(prompt))

    sources = [
        {
            "file": doc.metadata.get("source", "").split("/")[-1],
            "page": doc.metadata.get("page", "?"),
        }
        for doc in docs
    ]

    return {"answer": answer, "sources": sources, "in_scope": True}

print("Guardrails initialized.")
print("  Layer 1: LLM scope classifier (IN_SCOPE / OUT_OF_SCOPE)")
print("  Layer 2: FAISS relevance score floor (< 0.25 -> refuse)")


Guardrails initialized.
  Layer 1: LLM scope classifier (IN_SCOPE / OUT_OF_SCOPE)
  Layer 2: FAISS relevance score floor (< 0.25 -> refuse)


## Cell 12 — Test the Bot

### Your Task

Test your RAG pipeline using a few sample questions of your choice.

In [16]:
test_questions = [
    "How many days of earned leave do employees get per year?",
    "What is the payroll cut-off date and when is salary credited?",
    "Can you recommend stocks I should invest in?",
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")
    result = ask_bot(q)
    print(f"  In scope : {result['in_scope']}")
    print(f"  Answer   : {result['answer'][:400]}")
    if result["sources"]:
        print(f"  Sources  : {result['sources']}")
    print("-" * 60)


Q1: How many days of earned leave do employees get per year?
  In scope : True
  Answer   : Employees become eligible for 15 days of Earned Leave upon completion of one year of continuous service. Thereafter, Earned Leave accrues at the rate of 1.25 days per month.
  Sources  : [{'file': '02_Leave_Policy.pdf', 'page': 1}, {'file': '02_Leave_Policy.pdf', 'page': 1}, {'file': '06_Compensation_and_Benefits_Policy.pdf', 'page': 0}, {'file': '02_Leave_Policy.pdf', 'page': 1}, {'file': '02_Leave_Policy.pdf', 'page': 2}]
------------------------------------------------------------
Q2: What is the payroll cut-off date and when is salary credited?
  In scope : True
  Answer   : The payroll cut-off date is the 24th of each month. Salaries are processed and credited to the employee's registered bank account by the 7th of the following month.
  Sources  : [{'file': '06_Compensation_and_Benefits_Policy.pdf', 'page': 0}, {'file': '06_Compensation_and_Benefits_Policy.pdf', 'page': 0}, {'file': '06_Co

## Cell 13 — LangSmith Trace URL

Generate and copy your shareable LangSmith trace URL for submission.

> This cell is pre-filled — just run it and follow the instructions.

In [17]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



## Cell 14 — Streamlit App

### Your Task

Build and deploy a Streamlit chatbot application for your RAG pipeline.

Save your implementation as `app.py`.

In [18]:
app_code = 'import streamlit as st\nimport os\nimport glob\n\n# ─── Page configuration ───────────────────────────────────────────────────────\nst.set_page_config(\n    page_title="Zyro Dynamics HR Help Desk",\n    page_icon="🏢",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n# ─── Custom CSS ───────────────────────────────────────────────────────────────\nst.markdown("""\n<style>\n    .main-header {\n        background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);\n        padding: 2rem;\n        border-radius: 12px;\n        margin-bottom: 1.5rem;\n        text-align: center;\n        color: white;\n    }\n    .main-header h1 { margin: 0; font-size: 2rem; }\n    .main-header p  { margin: 0.5rem 0 0; opacity: 0.85; font-size: 1rem; }\n    .source-badge {\n        display: inline-block;\n        background: #e8f4fd;\n        border: 1px solid #bee3f8;\n        border-radius: 6px;\n        padding: 2px 10px;\n        font-size: 0.78rem;\n        color: #2b6cb0;\n        margin: 2px;\n    }\n    .oos-box {\n        background: #fff5f5;\n        border-left: 4px solid #fc8181;\n        padding: 0.75rem 1rem;\n        border-radius: 4px;\n        font-size: 0.9rem;\n    }\n</style>\n""", unsafe_allow_html=True)\n\n# ─── Header ───────────────────────────────────────────────────────────────────\nst.markdown("""\n<div class="main-header">\n    <h1>🏢 Zyro Dynamics HR Help Desk</h1>\n    <p>Ask any question about our HR policies — leave, compensation, WFH, POSH, and more.</p>\n</div>\n""", unsafe_allow_html=True)\n\n# ─── Sidebar ──────────────────────────────────────────────────────────────────\nwith st.sidebar:\n    st.header("⚙️ Configuration")\n    groq_key = st.text_input(\n        "Groq API Key",\n        type="password",\n        value=os.environ.get("GROQ_API_KEY", ""),\n        help="Get a free key at https://console.groq.com",\n    )\n    langchain_key = st.text_input(\n        "LangChain API Key (optional)",\n        type="password",\n        value=os.environ.get("LANGCHAIN_API_KEY", ""),\n        help="Enables LangSmith tracing",\n    )\n\n    st.divider()\n    st.header("💡 Sample Questions")\n    sample_qs = [\n        "How many days of earned leave do I get per year?",\n        "When is salary credited each month?",\n        "What is the WFH policy for L3 employees?",\n        "What health insurance coverage do I get?",\n        "What happens if I get a rating of 1 in my review?",\n        "How long is the PIP duration?",\n        "When is the Annual Performance Review?",\n        "What is the payroll cut-off date?",\n    ]\n    for sq in sample_qs:\n        if st.button(sq, use_container_width=True, key=f"sq_{sq[:20]}"):\n            st.session_state.pending_question = sq\n\n    st.divider()\n    if st.button("🗑️ Clear Chat", use_container_width=True):\n        st.session_state.messages = []\n\n    st.divider()\n    st.caption(\n        "**Zyro Dynamics HR Help Desk**  \\n"\n        "Powered by LangChain · FAISS · Groq · Streamlit  \\n"\n        "Built for the NIAT × Kaggle RAG Challenge"\n    )\n\n# ─── RAG pipeline (cached — builds only once per session) ────────────────────\n@st.cache_resource(show_spinner="📚 Building HR knowledge base…")\ndef build_pipeline(_groq_key: str):\n    from langchain_community.document_loaders import PyPDFLoader\n    from langchain_text_splitters import RecursiveCharacterTextSplitter\n    from langchain_huggingface import HuggingFaceEmbeddings\n    from langchain_community.vectorstores import FAISS\n    from langchain_groq import ChatGroq\n    from langchain_core.prompts import ChatPromptTemplate\n    from langchain_core.output_parsers import StrOutputParser\n\n    # ── Find corpus ───────────────────────────────────────────────────────────\n    # Kaggle path first, then same directory as app.py\n    kaggle_path = "/kaggle/input/zyro-dynamics-hr-corpus/"\n    local_path  = os.path.dirname(os.path.abspath(__file__))\n    corpus_path = kaggle_path if os.path.exists(kaggle_path) else local_path\n\n    pdf_files = sorted(glob.glob(os.path.join(corpus_path, "*.pdf")))\n    if not pdf_files:\n        st.error(f"No PDF files found in {corpus_path}. Make sure the dataset is attached.")\n        st.stop()\n\n    # ── Load → chunk → embed → index ─────────────────────────────────────────\n    documents = []\n    for pdf_path in pdf_files:\n        loader = PyPDFLoader(pdf_path)\n        documents.extend(loader.load())\n\n    splitter = RecursiveCharacterTextSplitter(\n        chunk_size=800,\n        chunk_overlap=120,\n        separators=["\\n\\n", "\\n", ". ", " ", ""],\n    )\n    chunks = splitter.split_documents(documents)\n\n    embeddings = HuggingFaceEmbeddings(\n        model_name="sentence-transformers/all-MiniLM-L6-v2",\n        model_kwargs={"device": "cpu"},\n        encode_kwargs={"normalize_embeddings": True},\n    )\n    vectorstore = FAISS.from_documents(chunks, embeddings)\n\n    # ── LLM ──────────────────────────────────────────────────────────────────\n    llm = ChatGroq(\n        model="llama-3.3-70b-versatile",\n        temperature=0.1,\n        max_tokens=512,\n        api_key=_groq_key,\n    )\n    parser = StrOutputParser()\n\n    # ── Prompts ───────────────────────────────────────────────────────────────\n    RAG_PROMPT = ChatPromptTemplate.from_template("""\nYou are an HR Help Desk assistant for Zyro Dynamics (also known as Acrux Dynamics).\nAnswer the employee\'s question using ONLY the information in the HR policy context below.\nBe specific: include exact numbers, dates, limits, and percentages when they appear in the context.\nDo NOT add information not found in the context.\nIf the context does not contain enough information, say:\n"I don\'t have enough information in the HR policy documents to answer this question."\n\nHR Policy Context:\n{context}\n\nEmployee Question: {question}\n\nAnswer:\n""")\n\n    OOS_PROMPT = ChatPromptTemplate.from_template("""\nYou are a strict classifier. Decide if the question can be answered using\nZyro Dynamics internal HR policy documents.\n\nHR policy documents cover ONLY:\n- Leave policies (earned, sick, maternity, paternity, etc.)\n- Work from home / hybrid / remote arrangements\n- Code of conduct and ethics\n- Performance reviews, PIP, APR, promotions\n- Compensation, salary bands, bonuses, payroll dates\n- Health insurance and employee benefits\n- IT and data security policies\n- Prevention of sexual harassment (POSH)\n- Onboarding, probation, separation, full & final settlement\n- Business travel and expense reimbursement\n- Company overview, grade structure, office policies\n\nQuestion: {question}\n\nReply with exactly one word — IN_SCOPE or OUT_OF_SCOPE. No explanation.\n""")\n\n    REFUSAL = (\n        "I\'m sorry, but I can only answer HR-related questions based on "\n        "Zyro Dynamics internal policy documents. Your question falls outside "\n        "the scope of what I can help with here. Please reach out to the relevant "\n        "department or consult an external resource."\n    )\n\n    def format_docs(docs):\n        parts = []\n        for i, doc in enumerate(docs, 1):\n            src  = doc.metadata.get("source", "").split("/")[-1]\n            page = doc.metadata.get("page", "?")\n            parts.append(f"[Source {i}: {src}, p.{page}]\\n{doc.page_content}")\n        return "\\n\\n---\\n\\n".join(parts)\n\n    def ask_bot(question: str) -> dict:\n        # Layer 1 — LLM scope classifier\n        verdict = parser.invoke(\n            llm.invoke(OOS_PROMPT.invoke({"question": question}))\n        ).strip().upper()\n        if "OUT_OF_SCOPE" in verdict:\n            return {"answer": REFUSAL, "sources": [], "in_scope": False}\n\n        # Layer 2 — similarity score floor\n        docs_scores = vectorstore.similarity_search_with_relevance_scores(question, k=5)\n        if docs_scores and docs_scores[0][1] < 0.25:\n            return {"answer": REFUSAL, "sources": [], "in_scope": False}\n\n        docs    = [d for d, _ in docs_scores]\n        context = format_docs(docs)\n        answer  = parser.invoke(\n            llm.invoke(RAG_PROMPT.invoke({"context": context, "question": question}))\n        )\n        sources = [\n            {"file": d.metadata.get("source", "").split("/")[-1],\n             "page": d.metadata.get("page", "?")}\n            for d in docs\n        ]\n        return {"answer": answer, "sources": sources, "in_scope": True}\n\n    return ask_bot\n\n# ─── Apply env vars from sidebar ─────────────────────────────────────────────\nif groq_key:\n    os.environ["GROQ_API_KEY"] = groq_key\nif langchain_key:\n    os.environ["LANGCHAIN_API_KEY"]    = langchain_key\n    os.environ["LANGCHAIN_TRACING_V2"] = "true"\n    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"\n\n# ─── Guard: require API key ───────────────────────────────────────────────────\nif not groq_key:\n    st.info(\n        "👈 **Enter your Groq API Key** in the sidebar to start.  \\n"\n        "Get a free key at [console.groq.com](https://console.groq.com).",\n        icon="🔑",\n    )\n    st.stop()\n\n# ─── Build pipeline ───────────────────────────────────────────────────────────\nask_bot = build_pipeline(groq_key)\n\n# ─── Session state init ───────────────────────────────────────────────────────\nif "messages" not in st.session_state:\n    st.session_state.messages = []\n\n# ─── Render chat history ─────────────────────────────────────────────────────\nfor msg in st.session_state.messages:\n    with st.chat_message(msg["role"]):\n        if not msg.get("in_scope", True):\n            st.markdown(\n                f\'<div class="oos-box">⚠️ {msg["content"]}</div>\',\n                unsafe_allow_html=True,\n            )\n        else:\n            st.write(msg["content"])\n        if msg.get("sources"):\n            with st.expander("📄 Policy sources", expanded=False):\n                for s in msg["sources"]:\n                    st.markdown(\n                        f\'<span class="source-badge">📑 {s["file"]} — page {s["page"]}</span>\',\n                        unsafe_allow_html=True,\n                    )\n\n# ─── Common handler for any question ─────────────────────────────────────────\ndef handle_question(prompt: str):\n    st.session_state.messages.append({"role": "user", "content": prompt})\n    with st.chat_message("user"):\n        st.write(prompt)\n    with st.chat_message("assistant"):\n        with st.spinner("🔍 Searching HR policies…"):\n            result   = ask_bot(prompt)\n        answer   = result["answer"]\n        sources  = result.get("sources", [])\n        in_scope = result.get("in_scope", True)\n        if not in_scope:\n            st.markdown(\n                f\'<div class="oos-box">⚠️ {answer}</div>\',\n                unsafe_allow_html=True,\n            )\n        else:\n            st.write(answer)\n        if sources:\n            with st.expander("📄 Policy sources", expanded=False):\n                for s in sources:\n                    st.markdown(\n                        f\'<span class="source-badge">📑 {s["file"]} — page {s["page"]}</span>\',\n                        unsafe_allow_html=True,\n                    )\n    st.session_state.messages.append({\n        "role": "assistant",\n        "content": answer,\n        "sources": sources,\n        "in_scope": in_scope,\n    })\n\n# ─── Handle sidebar sample click ─────────────────────────────────────────────\nif "pending_question" in st.session_state:\n    handle_question(st.session_state.pop("pending_question"))\n\n# ─── Live chat input ─────────────────────────────────────────────────────────\nif prompt := st.chat_input("Ask an HR question…"):\n    handle_question(prompt)\n'

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py written successfully.")
print("\nDeploy steps:")
print("  1. In Kaggle: File > Open in > add output to dataset OR download app.py")
print("  2. Push app.py to a GitHub repo (public)")
print("  3. Go to https://share.streamlit.io > New app > select your repo")
print("  4. Set GROQ_API_KEY and LANGCHAIN_API_KEY as Streamlit secrets")
print("  5. Copy the deployed *.streamlit.app URL for submission")


app.py written successfully.

Deploy steps:
  1. In Kaggle: File > Open in > add output to dataset OR download app.py
  2. Push app.py to a GitHub repo (public)
  3. Go to https://share.streamlit.io > New app > select your repo
  4. Set GROQ_API_KEY and LANGCHAIN_API_KEY as Streamlit secrets
  5. Copy the deployed *.streamlit.app URL for submission


## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [19]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [ ]:
import re

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL: ").strip()
langsmith_link = input("LangSmith Trace URL: ").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator


## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [ ]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")